## **I. Libraries and Data**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import joblib

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [9]:
df=pd.read_csv("housing-prices-dataset.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   price             545 non-null    int64
 1   area              545 non-null    int64
 2   bedrooms          545 non-null    int64
 3   bathrooms         545 non-null    int64
 4   stories           545 non-null    int64
 5   mainroad          545 non-null    str  
 6   guestroom         545 non-null    str  
 7   basement          545 non-null    str  
 8   hotwaterheating   545 non-null    str  
 9   airconditioning   545 non-null    str  
 10  parking           545 non-null    int64
 11  prefarea          545 non-null    str  
 12  furnishingstatus  545 non-null    str  
dtypes: int64(6), str(7)
memory usage: 55.5 KB


## **II. Data Exploratory**

In [10]:
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [11]:
df.shape

(545, 13)

In [12]:
df.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [13]:
df.columns

Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus'],
      dtype='str')

In [14]:
df["price"].unique()

array([13300000, 12250000, 12215000, 11410000, 10850000, 10150000,
        9870000,  9800000,  9681000,  9310000,  9240000,  9100000,
        8960000,  8890000,  8855000,  8750000,  8680000,  8645000,
        8575000,  8540000,  8463000,  8400000,  8295000,  8190000,
        8120000,  8080940,  8043000,  7980000,  7962500,  7910000,
        7875000,  7840000,  7700000,  7560000,  7525000,  7490000,
        7455000,  7420000,  7350000,  7343000,  7245000,  7210000,
        7140000,  7070000,  7035000,  7000000,  6930000,  6895000,
        6860000,  6790000,  6755000,  6720000,  6685000,  6650000,
        6629000,  6615000,  6580000,  6510000,  6475000,  6440000,
        6419000,  6405000,  6300000,  6293000,  6265000,  6230000,
        6195000,  6160000,  6125000,  6107500,  6090000,  6083000,
        6020000,  5950000,  5943000,  5880000,  5873000,  5866000,
        5810000,  5803000,  5775000,  5740000,  5652500,  5600000,
        5565000,  5530000,  5523000,  5495000,  5460000,  5425

## **III. Data Wrangling**

### **1. Outlier Detect:**

In [16]:
def detect_outliers(df, column_name):
    """
    Parameters:
    - df: DataFrame containing the data
    - column_name: Name of the column to check for outliers

    Returns:
    - Tuple (original DataFrame, DataFrame containing outliers, lower_bound, upper_bound)
    """
    # Create a copy of the DataFrame to avoid modifying the original data
    df_processed = df.copy()

    # Calculate Q1 (first quartile), Q3 (third quartile), and IQR
    Q1 = df_processed[column_name].quantile(0.25)
    Q3 = df_processed[column_name].quantile(0.75)
    IQR = Q3 - Q1

    # Determine lower and upper bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Detect outliers
    outliers = df_processed[(df_processed[column_name] < lower_bound) |
                          (df_processed[column_name] > upper_bound)]

    return df_processed, outliers, lower_bound, upper_bound


### **2. Handle Outliers**

In [17]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
# Replace with outlier handling
# Function to handle outliers
def handle_outliers(df, column_name, lower_bound, upper_bound, method='remove'):
    """
    Parameters:
    - df: DataFrame containing the data
    - column_name: Name of the column to handle outliers
    - lower_bound: Lower bound
    - upper_bound: Upper bound
    - method: 'remove' (drop) or 'replace' (cap with boundary values)

    Returns:
    - Processed DataFrame
    """
    # Create a copy of the DataFrame to avoid modifying the original data
    df_processed = df.copy()

    if method == 'remove':
        # Remove outliers
        df_processed = df_processed[(df_processed[column_name] >= lower_bound) &
                                  (df_processed[column_name] <= upper_bound)]
        print(f"Removed {len(df) - len(df_processed)} outliers.")

    elif method == 'replace':
        # Replace outliers with boundary values (lower_bound or upper_bound)
        df_processed.loc[df_processed[column_name] < lower_bound, column_name] = lower_bound
        df_processed.loc[df_processed[column_name] > upper_bound, column_name] = upper_bound
        print(f"Replaced {len(df) - len(df_processed[df_processed[column_name].between(lower_bound, upper_bound)])} outliers with boundary values.")

    else:
        raise ValueError("Method must be either 'remove' or 'replace'.")

    return df_processed


### **3. Check with Data**

In [19]:
column_name = "price"
df_processed, outliers, lower_bound, upper_bound = detect_outliers(df, column_name)
print("\nDetected outliers:")
print(outliers[column_name])
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")


Detected outliers:
0     13300000
1     12250000
2     12250000
3     12215000
4     11410000
5     10850000
6     10150000
7     10150000
8      9870000
9      9800000
10     9800000
11     9681000
12     9310000
13     9240000
14     9240000
Name: price, dtype: int64
Lower bound: -35000.0, Upper bound: 9205000.0


In [20]:
column_name = "price"
df_cleaned = handle_outliers(df, column_name, lower_bound, upper_bound, method='remove')
print("\nData after removing outliers:")
df_cleaned.info()

Removed 15 outliers.

Data after removing outliers:
<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 15 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   price             530 non-null    int64
 1   area              530 non-null    int64
 2   bedrooms          530 non-null    int64
 3   bathrooms         530 non-null    int64
 4   stories           530 non-null    int64
 5   mainroad          530 non-null    str  
 6   guestroom         530 non-null    str  
 7   basement          530 non-null    str  
 8   hotwaterheating   530 non-null    str  
 9   airconditioning   530 non-null    str  
 10  parking           530 non-null    int64
 11  prefarea          530 non-null    str  
 12  furnishingstatus  530 non-null    str  
dtypes: int64(6), str(7)
memory usage: 54.0 KB


In [21]:
column_name = "area"
df_processed, outliers, lower_bound, upper_bound = detect_outliers(df, column_name)
print("\nDetected outliers:")
print(outliers[column_name])
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")


Detected outliers:
7      16200
10     13200
56     11440
64     11175
66     13200
69     12090
125    15600
129    11460
186    11410
191    10700
211    12900
403    12944
Name: area, dtype: int64
Lower bound: -540.0, Upper bound: 10500.0


In [22]:
column_name = "area"
df_cleaned = handle_outliers(df_cleaned, column_name, lower_bound, upper_bound, method='remove')
print("\nData after removing outliers:")
df_cleaned.info()

Removed 10 outliers.

Data after removing outliers:
<class 'pandas.DataFrame'>
Index: 520 entries, 15 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   price             520 non-null    int64
 1   area              520 non-null    int64
 2   bedrooms          520 non-null    int64
 3   bathrooms         520 non-null    int64
 4   stories           520 non-null    int64
 5   mainroad          520 non-null    str  
 6   guestroom         520 non-null    str  
 7   basement          520 non-null    str  
 8   hotwaterheating   520 non-null    str  
 9   airconditioning   520 non-null    str  
 10  parking           520 non-null    int64
 11  prefarea          520 non-null    str  
 12  furnishingstatus  520 non-null    str  
dtypes: int64(6), str(7)
memory usage: 56.9 KB


### **4. Deal with NaN**

In [23]:
df.dropna(subset=['price'], inplace=True)